# AirShift — XGBoost Model Tuning

This notebook focuses on improving the XGBoost model selected during the model comparison stage.

The tuning process will explore different XGBoost hyperparameters and compare the tuned model with the baseline XGBoost model.

The final test period will remain completely unseen during tuning and will only be used for the final evaluation.


## 1. Load the Labeled Dataset

The labeled dataset is loaded from the processed data directory.

This dataset contains the engineered features and the binary deterioration target created in the previous stages of the AirShift pipeline.


In [14]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier


In [11]:
DATA_PATH = Path("../data/processed/airshift_labeled.csv")

df = pd.read_csv(DATA_PATH)

df["datetime"] = pd.to_datetime(df["datetime"])

print("Dataset shape:", df.shape)
print("Number of stations:", df["station"].nunique())
print("Date range:", df["datetime"].min(), "to", df["datetime"].max())

Dataset shape: (418381, 100)
Number of stations: 12
Date range: 2013-03-01 00:00:00 to 2017-02-28 17:00:00


In [2]:
print("\nTarget distribution:")
print(df["Deterioration"].value_counts())

print("\nTarget proportions:")
print(df["Deterioration"].value_counts(normalize=True).round(4))


Target distribution:
Deterioration
0.0    219985
1.0    198396
Name: count, dtype: int64

Target proportions:
Deterioration
0.0    0.5258
1.0    0.4742
Name: proportion, dtype: float64


In [3]:
# Quick validation
print("Missing target values:", df["Deterioration"].isna().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing target values: 0
Duplicate rows: 0


## 2. Define Features and Target

The deterioration label is used as the target variable, while identifier and datetime columns are excluded from the model features.

The `datetime` column is retained separately for creating the chronological data splits.


In [4]:
TARGET = "Deterioration"

EXCLUDED_COLUMNS = [
    "Deterioration",
    "No",
    "datetime"
]

X = df.drop(columns=EXCLUDED_COLUMNS)
y = df[TARGET]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Number of features:", X.shape[1])

Feature matrix shape: (418381, 97)
Target shape: (418381,)
Number of features: 97


In [5]:
print("Categorical features:")
print(X.select_dtypes(include=["object"]).columns.tolist())

print("\nNumerical features:")
print(X.select_dtypes(exclude=["object"]).columns.tolist())

Categorical features:
['wd', 'station']

Numerical features:
['year', 'month', 'day', 'hour', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'day_of_week', 'is_weekend', 'PM2.5_lag_1h', 'PM2.5_lag_3h', 'PM2.5_lag_6h', 'PM10_lag_1h', 'PM10_lag_3h', 'PM10_lag_6h', 'SO2_lag_1h', 'SO2_lag_3h', 'SO2_lag_6h', 'NO2_lag_1h', 'NO2_lag_3h', 'NO2_lag_6h', 'CO_lag_1h', 'CO_lag_3h', 'CO_lag_6h', 'O3_lag_1h', 'O3_lag_3h', 'O3_lag_6h', 'PM2.5_rolling_mean_3h', 'PM2.5_rolling_max_3h', 'PM2.5_rolling_std_3h', 'PM2.5_rolling_mean_6h', 'PM2.5_rolling_max_6h', 'PM2.5_rolling_std_6h', 'PM10_rolling_mean_3h', 'PM10_rolling_max_3h', 'PM10_rolling_std_3h', 'PM10_rolling_mean_6h', 'PM10_rolling_max_6h', 'PM10_rolling_std_6h', 'SO2_rolling_mean_3h', 'SO2_rolling_max_3h', 'SO2_rolling_std_3h', 'SO2_rolling_mean_6h', 'SO2_rolling_max_6h', 'SO2_rolling_std_6h', 'NO2_rolling_mean_3h', 'NO2_rolling_max_3h', 'NO2_rolling_std_3h', 'NO2_rolling_mean_6h', 'NO2_rolling_max_6h', 'NO2_ro

C:\Users\HP\AppData\Local\Temp\ipykernel_6840\2682472949.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(X.select_dtypes(include=["object"]).columns.tolist())


### Findings

The labeled dataset contains **97 features** after excluding the target, record identifier, and datetime columns. The features include **95 numerical variables** and **2 categorical variables** (`wd` and `station`).


## 3. Temporal Data Split

The data is divided chronologically into model-training, validation, and final test periods.

The model-training period is used to train the XGBoost model, while the validation period is used for hyperparameter tuning and model selection. The final test period remains completely unseen until the final evaluation.

This chronological split prevents future information from being used during model development.


### 3.1 Define the periods

In [6]:
MODEL_TRAIN_END = "2014-12-31 23:00:00"
VALIDATION_START = "2015-01-01 00:00:00"
VALIDATION_END = "2015-12-31 23:00:00"
TEST_START = "2016-01-01 00:00:00"

model_train_mask = df["datetime"] <= MODEL_TRAIN_END

validation_mask = (
    (df["datetime"] >= VALIDATION_START)
    & (df["datetime"] <= VALIDATION_END)
)

test_mask = df["datetime"] >= TEST_START

### 3.2 Create the splits

In [7]:
model_train_df = df.loc[model_train_mask].copy()
validation_df = df.loc[validation_mask].copy()
test_df = df.loc[test_mask].copy()

print("Model training period:")
print(model_train_df["datetime"].min(), "to", model_train_df["datetime"].max())
print("Observations:", len(model_train_df))

print("\nValidation period:")
print(validation_df["datetime"].min(), "to", validation_df["datetime"].max())
print("Observations:", len(validation_df))

print("\nFinal test period:")
print(test_df["datetime"].min(), "to", test_df["datetime"].max())
print("Observations:", len(test_df))

Model training period:
2013-03-01 00:00:00 to 2014-12-31 23:00:00
Observations: 191665

Validation period:
2015-01-01 00:00:00 to 2015-12-31 23:00:00
Observations: 104816

Final test period:
2016-01-01 00:00:00 to 2017-02-28 17:00:00
Observations: 121900


### 3.3 Prepare X and y

In [8]:
X_model_train = model_train_df.drop(columns=EXCLUDED_COLUMNS)
y_model_train = model_train_df[TARGET]

X_validation = validation_df.drop(columns=EXCLUDED_COLUMNS)
y_validation = validation_df[TARGET]

X_test = test_df.drop(columns=EXCLUDED_COLUMNS)
y_test = test_df[TARGET]

print("X_model_train:", X_model_train.shape)
print("y_model_train:", y_model_train.shape)

print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_model_train: (191665, 97)
y_model_train: (191665,)
X_validation: (104816, 97)
y_validation: (104816,)
X_test: (121900, 97)
y_test: (121900,)


### 3.4 Verify temporal separation

In [9]:
print(
    "Training ends:",
    model_train_df["datetime"].max()
)

print(
    "Validation starts:",
    validation_df["datetime"].min()
)

print(
    "Validation ends:",
    validation_df["datetime"].max()
)

print(
    "Test starts:",
    test_df["datetime"].min()
)

Training ends: 2014-12-31 23:00:00
Validation starts: 2015-01-01 00:00:00
Validation ends: 2015-12-31 23:00:00
Test starts: 2016-01-01 00:00:00


### Findings

The data was split chronologically into **191,665 training observations**, **104,816 validation observations**, and **121,900 final test observations**.

The training, validation, and test periods are strictly separated in time, ensuring that future observations are not used during model development.


## 4. XGBoost Preprocessing

The preprocessing pipeline prepares the numerical and categorical features for XGBoost.

Missing numerical values are replaced using the median, while missing categorical values are replaced using the most frequent category. Categorical variables are one-hot encoded.

Numerical features are not standardized because XGBoost is a tree-based model and does not require feature scaling.


In [12]:
categorical_features = ["wd", "station"]

numerical_features = [
    col for col in X_model_train.columns
    if col not in categorical_features
]

print("Categorical features:", categorical_features)
print("Number of numerical features:", len(numerical_features))

Categorical features: ['wd', 'station']
Number of numerical features: 95


In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),
        (
            "numerical",
            SimpleImputer(strategy="median"),
            numerical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## 5. Baseline XGBoost

A baseline XGBoost model is trained using the initial hyperparameter configuration selected during the model development stage.

The baseline provides a reference point for evaluating whether hyperparameter tuning improves model performance.

The model is trained only on the model-training period, while the validation period remains unseen during training.


In [15]:
baseline_xgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=6,
                random_state=42,
                n_jobs=-1,
                eval_metric="logloss"
            )
        )
    ]
)

print("Baseline XGBoost pipeline created successfully.")

Baseline XGBoost pipeline created successfully.


In [16]:
import time

start_time = time.time()

baseline_xgb.fit(
    X_model_train,
    y_model_train
)

training_time = time.time() - start_time

print(f"Baseline XGBoost training time: {training_time:.2f} seconds")

Baseline XGBoost training time: 35.04 seconds
